# Solution to p10

The upsampled decoder must match the skip in batch and both spatial coordinates. Concatenating decoder first and skip second on channels makes channels 4–7 exactly the skip tensor.

In [ ]:
import torch
from torch import nn

bottleneck = torch.arange(2*8*4*4, dtype=torch.float32).reshape(2,8,4,4) / 100.0
skip = torch.arange(2*4*8*8, dtype=torch.float32).reshape(2,4,8,8) / 200.0
up = nn.ConvTranspose2d(8, 4, 2, 2, bias=True)
fuse = nn.Conv2d(8, 4, 3, padding=1, bias=True)
head = nn.Conv2d(4, 2, 1, bias=True)
with torch.no_grad():
    for module in (up, fuse, head):
        module.weight.fill_(0.01)
        module.bias.zero_()
ATOL = 1e-6
RTOL = 1e-6

def decode_with_skip(bottleneck, skip, up, fuse, head):
    if bottleneck.ndim != 4 or skip.ndim != 4:
        raise ValueError("decoder inputs must be 4-D")
    if bottleneck.dtype != torch.float32 or skip.dtype != torch.float32:
        raise ValueError("decoder inputs must be float32")
    upsampled = up(bottleneck)
    if upsampled.shape[0] != skip.shape[0] or upsampled.shape[2:] != skip.shape[2:]:
        raise ValueError("batch and spatial shapes must match exactly")
    concatenated = torch.cat((upsampled, skip), dim=1)
    logits = head(fuse(concatenated))
    return logits, concatenated

logits, concatenated = decode_with_skip(bottleneck, skip, up, fuse, head)


### Answer check

In [ ]:
assert up(bottleneck).shape == (2,4,8,8)
assert concatenated.shape == (2,8,8,8)
assert logits.shape == (2,2,8,8)
torch.testing.assert_close(concatenated[:,4:8], skip, atol=ATOL, rtol=RTOL)
try:
    decode_with_skip(bottleneck, skip[:,:,:7,:], up, fuse, head)
except ValueError:
    pass
else:
    raise AssertionError("spatial mismatch was not rejected")
